[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maxischa/datacamp_test/blob/main/bloc4_ml/corrections/seance1_correction.ipynb)

# Séance 4.1 — Prédire un nombre — expliquer n'est pas prédire

**Correction** · durée : 2h (≈50 min de cours, ≈50 min d'exercices)

> ⚠️ **Avant de taper quoi que ce soit :** *Fichier → Enregistrer une copie dans Drive*. Sinon votre travail sera perdu en fermant l'onglet.
>
> 📱 Sur tablette, faites d'abord les réglages de [Bien démarrer](https://github.com/maxischa/datacamp_test/blob/main/ressources/setup_tablette.md).

## Objectifs

À la fin de cette séance, vous saurez :

- dire ce qui sépare un modèle qui explique d'un modèle qui prédit
- découper un jeu de données en apprentissage et test, et dire pourquoi
- mesurer une erreur de prédiction en euros avec la RMSE et la MAE
- reconnaître un surapprentissage à l'écart entre les deux jeux
- repérer une fuite de données — l'erreur qui donne un modèle parfait et inutile

## Correction

Solutions commentées. Comparez avec ce que vous aviez écrit : plusieurs formulations peuvent être correctes.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Affichage adapte aux petits ecrans
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 80)

# Les donnees sont lues directement depuis le web : rien a telecharger
BASE = "https://raw.githubusercontent.com/maxischa/datacamp_test/main/bloc4_ml/data/"

In [ ]:
def verifier(nom, condition, indice=""):
    """Affiche un retour immediat sans interrompre le notebook."""
    print("OK   -", nom) if condition else print("A REVOIR -", nom, ":", indice)

Chargement des données utilisées dans toute la feuille :

In [ ]:
cmd = pd.read_csv(BASE + "commandes.csv")

X = cmd[["qte", "nart"]]   ## ce qu'on connait avant de facturer
y = cmd["ca"]              ## ce qu'on veut prevoir

# test_size=0.25 : un quart des lignes mis de cote pour la notation
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=67)
print(len(X_tr), "commandes d'apprentissage,", len(X_te), "de test")

---

# Partie 1 — L'échauffement

Le code est déjà écrit : il ne reste que les `____` à remplir. Allez vite, l'essentiel
de la séance est dans la partie 2.

### Exercice 1 — Ajuster et prédire

> **Votre mission :**
> - Ajuster une régression linéaire sur le jeu d'**apprentissage** → `m`.
> - Prédire sur le jeu de **test** → `p`.
> - Mettre la première prédiction, arrondie à 2 décimales, dans `p1`.

In [ ]:
# On apprend sur X_tr / y_tr, on predit sur X_te : jamais l'inverse
m = LinearRegression().fit(X_tr, y_tr)   ## apprendre
p = m.predict(X_te)                      ## noter, sur des lignes jamais vues

p1 = round(p[0], 2)   ## une prediction par ligne de test
print(len(p), "predictions | la premiere :", p1)

In [ ]:
verifier("1a - nombre de predictions", len(p) == 489, "on predit sur le jeu de test")
verifier("1b - premiere prediction", abs(p1 - 370.99) < 1,
         "ajustez sur X_tr et y_tr, puis predisez sur X_te")

### Exercice 2 — L'erreur en euros

> **Votre mission :**
> - Calculer la MAE et la RMSE du modèle **sur le jeu de test** → `mae` et `rmse`, arrondies à 1 décimale.
> - Rappel : la RMSE est la racine de `mean_squared_error`.

In [ ]:
mae = round(mean_absolute_error(y_te, p), 1)   ## en euros

# ** 0.5 : la racine carree. RMSE = Root Mean Squared Error
rmse = round(mean_squared_error(y_te, p) ** 0.5, 1)   ## punit les gros ecarts

print("MAE", mae, "euros | RMSE", rmse, "euros")

In [ ]:
verifier("2a - MAE", abs(mae - 217.5) < 1, "mean_absolute_error(y_te, p)")
verifier("2b - RMSE", abs(rmse - 583.3) < 1, "la racine carree s'ecrit ** 0.5")

### Exercice 3 — Les deux notes du même modèle

> **Votre mission :**
> - Calculer le R² du modèle **en apprentissage** → `r2_tr`, et **en test** → `r2_te`, arrondis à 3 décimales.
> - Lequel des deux est le meilleur ? De combien ? Cet écart est-il inquiétant ?

In [ ]:
r2_tr = round(r2_score(y_tr, m.predict(X_tr)), 3)   ## sur le vu
r2_te = round(r2_score(y_te, m.predict(X_te)), 3)   ## sur le jamais vu

print("apprentissage", r2_tr, "| test", r2_te)
print("ecart :", round(r2_tr - r2_te, 3))

# L'apprentissage est meilleur, comme toujours : les coefficients ont ete
# calcules pour coller a CES lignes-la. Quatre points d'ecart, c'est le
# prix normal a payer. L'exercice 4 montre a quoi ressemble un ecart qui,
# lui, doit alerter.

In [ ]:
verifier("3a - R2 en apprentissage", abs(r2_tr - 0.732) < 0.01, "predisez sur X_tr")
verifier("3b - R2 en test", abs(r2_te - 0.691) < 0.01, "predisez sur X_te")

### Exercice 4 — L'arbre qui apprend par cœur

> **Votre mission :**
> - Ajuster un `DecisionTreeRegressor` **sans limite de profondeur** (`random_state=42`) → `libre`.
> - Relever son R² en apprentissage → `r2_libre_tr`, et en test → `r2_libre_te`.

In [ ]:
libre = DecisionTreeRegressor(random_state=42).fit(X_tr, y_tr)

r2_libre_tr = round(r2_score(y_tr, libre.predict(X_tr)), 3)   ## 0,98 : par coeur
r2_libre_te = round(r2_score(y_te, libre.predict(X_te)), 3)   ## 0,50 : la verite
print("apprentissage", r2_libre_tr, "| test", r2_libre_te)

# 0,98 puis 0,50 : quarante-huit points d'ecart, contre quatre pour la
# regression. C'est le manuel du surapprentissage.

In [ ]:
verifier("4a - R2 en apprentissage", r2_libre_tr > 0.9,
         "un arbre sans limite reproduit presque parfaitement ce qu'il a vu")
verifier("4b - R2 en test", r2_libre_te < 0.6,
         "predisez sur le jeu de test, pas sur celui d'apprentissage")

### Exercice 5 — Brider pour améliorer

> **Votre mission :**
> - Ajuster le même arbre avec `max_depth=3` → `court`, et relever son R² **en test** → `r2_court`.
> - Comparer à celui de l'arbre libre.

In [ ]:
# max_depth=3 : on interdit a l'arbre de descendre plus bas
court = DecisionTreeRegressor(max_depth=3, random_state=42).fit(X_tr, y_tr)
r2_court = round(r2_score(y_te, court.predict(X_te)), 3)

print("libre", r2_libre_te, "| bride", r2_court)

# Le modele le plus SIMPLE predit le mieux. Contre-intuitif, et c'est
# pourtant la regle generale plutot que l'exception.

In [ ]:
verifier("5 - l'arbre bride fait mieux", r2_court > r2_libre_te,
         "max_depth=3 limite la profondeur de l'arbre")

### Exercice 6 — Prédire une commande qui arrive

> **Votre mission :**
> - Une commande arrive : **150 unités** (`qte`), **12 produits distincts** (`nart`).
> - Prédire son montant avec `m` → `devis`, arrondi à 2 décimales.
> - `predict` attend un tableau dont les colonnes portent les mêmes noms que `X`.

In [ ]:
# Les colonnes doivent porter les memes noms QUE DANS LE MEME ORDRE
nouvelle = pd.DataFrame({"qte": [150], "nart": [12]})   ## une seule ligne
devis = round(m.predict(nouvelle)[0], 2)   ## [0] : la premiere prediction

print("montant prevu :", devis, "euros")

In [ ]:
verifier("6 - montant prevu", abs(devis - 305.12) < 2,
         "la seconde colonne s'appelle nart")

### Exercice 7 — La fuite de données

> **Votre mission :**
> - Refaire un modèle en mettant `ca` parmi les variables explicatives → `r2_fuite`, arrondi à 3 décimales.
> - Le score est parfait. Expliquez en une phrase pourquoi ce modèle ne vaut rien.

In [ ]:
triche = cmd[["qte", "nart", "ca"]]   ## "ca" est la cible !
Xf_tr, Xf_te, yf_tr, yf_te = train_test_split(triche, y, test_size=0.25, random_state=67)

r2_fuite = round(r2_score(yf_te, LinearRegression().fit(Xf_tr, yf_tr).predict(Xf_te)), 3)
print(r2_fuite)

# On a donne la reponse au modele. Le jour ou une vraie commande arrive,
# la colonne ca n'existe pas encore : le modele est inutilisable.

In [ ]:
verifier("7 - modele parfait et inutile", r2_fuite == 1.0,
         "la colonne qui contient la reponse s'appelle ca")

### Exercice 8 — Question de synthèse

> **Votre mission :**
> - Le directeur logistique veut savoir s'il peut se fier au modèle pour dimensionner ses préparations.
> - Calculer l'erreur médiane absolue en euros → `err_med`, arrondie à 2 décimales.
> - La comparer à la médiane des commandes → `med_ca`. Que lui répondez-vous ?

In [ ]:
erreurs = (y_te - p).abs()

# La mediane des erreurs, plutot que leur moyenne : meme raisonnement
# qu'en seance 3.1, quelques grosses fautes tirent la moyenne
err_med = round(erreurs.median(), 2)   ## l'erreur d'une fois sur deux
med_ca = round(y_te.median(), 2)       ## a comparer a la commande typique
print("erreur mediane", err_med, "euros pour une commande mediane de", med_ca)

# Une reponse possible :
# "Sur une commande mediane de 343 EUR, nous nous trompons de 91 EUR une
#  fois sur deux, soit 27 %. C'est utilisable pour dimensionner un stock
#  global, pas pour engager un devis client."

In [ ]:
verifier("8a - erreur mediane", abs(err_med - 90.92) < 3, "la methode s'appelle median()")
verifier("8b - commande mediane", abs(med_ca - 342.90) < 3, "median() sur y_te")

---

# Partie 2 — Les questions

Ici, plus de trous : **la cellule sous chaque question est vide**, et c'est à vous
d'écrire le code en entier. C'est exactement ce qu'on vous demandera pour le projet
final, et ce que fait un analyste devant un fichier qu'il découvre.

Certaines questions utilisent une commande que le cours n'a pas montrée. Quand c'est le
cas, l'énoncé vous la donne — savoir se servir d'une commande qu'on vient de lire fait
partie du métier.

> 💡 Pas de vérification automatique dans cette partie. Affichez systématiquement votre
> résultat, et demandez-vous s'il est **plausible** avant de passer à la suite : c'est
> le seul contrôle dont vous disposerez en entreprise.

### Question 9 — Le découpage change-t-il la note ?

> **Votre mission :**
> - Refaire le découpage et l'évaluation avec cinq `random_state` différents (0, 1, 2, 3, 4).
> - Afficher le R² de test à chaque fois. De combien varie-t-il ?
> - Qu'en concluez-vous sur un score annoncé sans précision de découpage ?

In [ ]:
for graine in range(5):
    # Meme fichier, meme modele : seul le tirage du test change
    a, b, c, d = train_test_split(X, y, test_size=0.25, random_state=graine)
    r2 = r2_score(d, LinearRegression().fit(a, c).predict(b))
    print(f"random_state={graine} : R2 = {r2:.3f}")

# Le R2 va de 0,45 a 0,74 selon le tirage : annoncer "R2 = 0,69" sans dire
# sur quel decoupage, c'est annoncer un chiffre qu'on ne peut pas
# reproduire. La validation croisee de la seance 4.3 repond a ce probleme.

### Question 10 — Le modèle le plus bête

> **Votre mission :**
> - Construire un modèle de référence qui prédit **toujours la même valeur** : la moyenne des montants d'apprentissage.
> - Calculer sa RMSE de test. Quel R² obtient-il ?
> - Tout modèle doit battre celui-là, sinon il ne sert à rien.

In [ ]:
bete = np.full(len(y_te), y_tr.mean())   ## toujours la meme reponse

print("RMSE du modele bete :", round(mean_squared_error(y_te, bete) ** 0.5, 1))
print("RMSE du vrai modele :", round(mean_squared_error(y_te, p) ** 0.5, 1))
print("R2 du modele bete   :", round(r2_score(y_te, bete), 3))

# Le R2 du modele constant vaut zero a un millieme pres (RMSE 1 051 EUR,
# contre 583 pour le vrai modele) : c'est
# exactement ce que le R2 mesure — le gain par rapport a "toujours la
# moyenne". Ce modele de reference est le premier a construire, toujours.

### Question 11 — Prédire le prix moyen plutôt que le montant

> **Votre mission :**
> - Créer une cible `prix_moyen` = `ca` / `qte`, et tenter de la prédire à partir de `nart`.
> - Le R² de test est mauvais. Est-ce le modèle qui est mauvais, ou la question ?

In [ ]:
cible = cmd["ca"] / cmd["qte"]   ## le prix moyen d'un article
a, b, c, d = train_test_split(cmd[["nart"]], cible, test_size=0.25, random_state=67)

r2 = r2_score(d, LinearRegression().fit(a, c).predict(b))
print("R2 :", round(r2, 3))

# La question. Le nombre de produits distincts ne contient aucune information sur
# le prix unitaire moyen : un modele ne peut pas inventer un signal qui
# n'est pas dans les donnees. Un mauvais R2 est parfois une reponse.

### Question 12 — Un découpage qui triche

> **Votre mission :**
> - Les commandes sont triées par date. Refaire le découpage **sans mélanger** (`shuffle=False`) : le test devient les dernières commandes.
> - Comparer le R² de test à celui obtenu avec mélange.
> - Lequel des deux découpages imite le mieux la vraie vie ?

In [ ]:
# shuffle=False : le test devient les dernieres commandes, pas un tirage
a, b, c, d = train_test_split(X, y, test_size=0.25, shuffle=False)
r2_temps = r2_score(d, LinearRegression().fit(a, c).predict(b))

print("decoupage aleatoire :", round(r2_score(y_te, p), 3))
print("decoupage temporel  :", round(r2_temps, 3))

# Le decoupage temporel est le seul honnete ici : en vrai, on predit
# l'avenir avec le passe, jamais l'inverse. Un decoupage aleatoire laisse
# le modele apprendre sur des commandes POSTERIEURES a celles qu'il note.

### Question 13 — La fuite discrète

> **Votre mission :**
> - La fuite du cours était grossière. En voici une réaliste : ajouter une colonne `remise`, qui vaut 5 % du montant facturé.
> - Rien dans son nom ne dit qu'elle contient la réponse — et pourtant.
> - Mesurer le R² de test et expliquer ce qui s'est passé.

In [ ]:
discret = cmd[["qte", "nart"]].copy()
discret["remise"] = (cmd["ca"] * 0.05).round(2)   ## 5 % du montant facture

a, b, c, d = train_test_split(discret, y, test_size=0.25, random_state=67)
print("R2 :", round(r2_score(d, LinearRegression().fit(a, c).predict(b)), 3))

# R2 = 1,0. La remise vaut 5 % du montant : c'est la reponse, ecrite
# autrement. Aucune colonne ne s'appelait "ca", et pourtant la fuite est
# totale. Le test a appliquer : cette colonne serait-elle disponible AVANT
# que la commande soit facturee ? Non. Elle n'a rien a faire la.

### Question 14 — Choisir la bonne métrique

> **Votre mission :**
> - Deux modèles : la régression linéaire `m`, et l'arbre bridé `court`.
> - Comparer leurs MAE et leurs RMSE de test.
> - Si l'entreprise craint surtout les **grosses** erreurs de devis, lequel choisir ? Et si elle veut minimiser l'erreur courante ?

In [ ]:
# Deux metriques, deux classements possibles : le choix est une decision
for nom, modele in {"lineaire": m, "arbre bride": court}.items():
    pred = modele.predict(X_te)
    print(f"{nom:<12} MAE {mean_absolute_error(y_te, pred):7.1f}  "
          f"RMSE {mean_squared_error(y_te, pred) ** 0.5:7.1f}")

# Les deux metriques ne classent pas forcement les modeles dans le meme
# ordre. La RMSE punit les grosses fautes : c'est elle qu'on regarde si
# une erreur de 2 000 EUR sur un devis coute bien plus cher que vingt
# erreurs de 100 EUR. Le choix de la metrique est une decision de gestion.

### Question 15 — Question de synthèse

> **Votre mission :**
> - On vous demande une note d'une page : *« peut-on automatiser le devis à partir du nombre d'unités et de produits distincts ? »*
> - Produire les trois chiffres qui fondent votre réponse : l'erreur typique en euros, sa part du montant médian, et la part des commandes prédites à moins de 20 % près.
> - Puis rédigez la recommandation en commentaire.

In [ ]:
erreurs = (y_te - p).abs()
ecart_relatif = erreurs / y_te   ## une erreur de 95 EUR ne pese pas pareil

print("erreur mediane      :", round(erreurs.median(), 2), "euros")
print("soit                :", round(100 * (erreurs.median() / y_te.median()), 1), "%")
print("commandes a +/- 20 %:", round(100 * (ecart_relatif < 0.20).mean(), 1), "%")

# Recommandation possible :
# "Le modele predit deux commandes sur cinq a moins de 20 % pres, et se
#  trompe de 91 EUR sur un panier median de 343 EUR. C'est suffisant pour
#  dimensionner un stock hebdomadaire, insuffisant pour engager un devis
#  client. Nous recommandons un usage interne uniquement, et la collecte
#  du type d'article — la variable qui manque visiblement au modele."